<a href="https://colab.research.google.com/github/brunakv/ciencia_dados_II/blob/main/ciencia_dados_II_bruna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pyspark

In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CreditoRuralBrasil") \
    .getOrCreate()

In [ ]:
import requests
from pyspark.sql import SparkSession

url = "https://olinda.bcb.gov.br/olinda/servico/SICOR/versao/v2/odata/RegiaoUF?$format=json"

response = requests.get(url)
response.raise_for_status()

dados = response.json()

df_spark = spark.createDataFrame(dados["value"])
df_spark.show()

+----------+---------+----------+------------------+----------+-------------------+---------------+-----------------+---------+------------------+--------------+--------+--------------+----------+--------+-------------+------------+------+
|AnoEmissao|Atividade|MesEmissao|QtdComercializacao|QtdCusteio|QtdIndustrializacao|QtdInvestimento|VlComercializacao|VlCusteio|VlIndustrializacao|VlInvestimento|cdEstado|cdFonteRecurso|cdPrograma|cdRegiao|cdSubPrograma|  nomeRegiao|nomeUF|
+----------+---------+----------+------------------+----------+-------------------+---------------+-----------------+---------+------------------+--------------+--------+--------------+----------+--------+-------------+------------+------+
|      2019|        1|        04|                 0|         0|                  2|              0|              0.0|      0.0|         1671615.0|           0.0|      19|          0201|      0001|       4|           53|         SUL|    PR|
|      2024|        2|        08|       

In [ ]:
df_spark.printSchema()

root
 |-- AnoEmissao: string (nullable = true)
 |-- Atividade: string (nullable = true)
 |-- MesEmissao: string (nullable = true)
 |-- QtdComercializacao: long (nullable = true)
 |-- QtdCusteio: long (nullable = true)
 |-- QtdIndustrializacao: long (nullable = true)
 |-- QtdInvestimento: long (nullable = true)
 |-- VlComercializacao: double (nullable = true)
 |-- VlCusteio: double (nullable = true)
 |-- VlIndustrializacao: double (nullable = true)
 |-- VlInvestimento: double (nullable = true)
 |-- cdEstado: string (nullable = true)
 |-- cdFonteRecurso: string (nullable = true)
 |-- cdPrograma: string (nullable = true)
 |-- cdRegiao: string (nullable = true)
 |-- cdSubPrograma: string (nullable = true)
 |-- nomeRegiao: string (nullable = true)
 |-- nomeUF: string (nullable = true)



In [ ]:
print(df_spark.count())

194269


In [ ]:
from pyspark.sql.functions import col

df_spark = df_spark.withColumn(
    "VL_TOTAL",
    col("VlCusteio") +
    col("VlInvestimento") +
    col("VlComercializacao") +
    col("VlIndustrializacao")
)

In [ ]:
# Análise 1: Estados com maior volume financiado

from pyspark.sql.functions import sum

# Agrupar por estado e somar o VL_TOTAL
volumes_por_estado = df_spark.groupBy("nomeUF").agg(sum("VL_TOTAL").alias("VolumeTotalFinanciado"))

# Ordenar por volume total financiado em ordem decrescente
volumes_por_estado_ordenado = volumes_por_estado.orderBy("VolumeTotalFinanciado", ascending=False)

# Mostrar os 10 principais estados
volumes_por_estado_ordenado.show(10)

top10 = volumes_por_estado_ordenado.limit(10).select("nomeUF").collect()

lista_estados = [linha["nomeUF"] for linha in top10]

texto = (
    "Os estados com maior volume de crédito rural são: "
    + ", ".join(lista_estados[:-1])
    + " e "
    + lista_estados[-1]
    + "."
)

print(texto)

+------+---------------------+
|nomeUF|VolumeTotalFinanciado|
+------+---------------------+
|    PR| 4.697590000026804E11|
|    RS| 4.457188080159595E11|
|    MG| 4.167102475847004E11|
|    MT| 3.416379541751395E11|
|    SP|    3.365321300218E11|
|    GO| 2.991407894084796E11|
|    SC| 1.899293774435098...|
|    MS| 1.832157760690202E11|
|    BA| 1.289228109823001...|
|    TO| 7.848694634336996E10|
+------+---------------------+
only showing top 10 rows
Os estados com maior volume de crédito rural são: PR, RS, MG, MT, SP, GO, SC, MS, BA e TO.


In [ ]:
# Análise 2: O crédito rural cresceu ou diminuiu ao longo do tempo?

from pyspark.sql.functions import sum

evolucao_anual = (
    df_spark
    .groupBy("AnoEmissao")
    .agg(sum("VL_TOTAL").alias("VolumeTotal"))
    .orderBy("AnoEmissao")
)

evolucao_anual.show()
dados = evolucao_anual.collect()

# Histórico anual
texto_historico = "A evolução do crédito rural ao longo dos anos foi a seguinte:\n\n"

for linha in dados:
    texto_historico += (
        f"- {linha['AnoEmissao']}: "
        f"R$ {linha['VolumeTotal']:,.2f}\n"
    )

print(texto_historico)

primeiro_ano = dados[0]["AnoEmissao"]
ultimo_ano = dados[-1]["AnoEmissao"]

primeiro_valor = dados[0]["VolumeTotal"]
ultimo_valor = dados[-1]["VolumeTotal"]

variacao = ((ultimo_valor - primeiro_valor) / primeiro_valor) * 100

texto_conclusao = (
    f"Entre {primeiro_ano} e {ultimo_ano}, o volume de crédito rural "
    f"variou de R$ {primeiro_valor:,.2f} para "
    f"R$ {ultimo_valor:,.2f}, representando uma variação "
    f"de {variacao:.2f}% no período analisado."
)

print(texto_conclusao)

+----------+--------------------+
|AnoEmissao|         VolumeTotal|
+----------+--------------------+
|      2013|1.393859855320399E11|
|      2014|1.644308828048699E11|
|      2015|1.541460630198899...|
|      2016|1.176240631413100...|
|      2017|1.670173488596699...|
|      2018|1.808869343161897...|
|      2019|1.789626140696308E11|
|      2020|2.070524867862803...|
|      2021|2.951795606411706...|
|      2022|3.628874406414095...|
|      2023|4.065409363054596...|
|      2024|3.799044211521499...|
|      2025|3.607499530022201E11|
|      2026|1.738754470995500...|
+----------+--------------------+

A evolução do crédito rural ao longo dos anos foi a seguinte:

- 2013: R$ 139,385,985,532.04
- 2014: R$ 164,430,882,804.87
- 2015: R$ 154,146,063,019.89
- 2016: R$ 117,624,063,141.31
- 2017: R$ 167,017,348,859.67
- 2018: R$ 180,886,934,316.19
- 2019: R$ 178,962,614,069.63
- 2020: R$ 207,052,486,786.28
- 2021: R$ 295,179,560,641.17
- 2022: R$ 362,887,440,641.41
- 2023: R$ 406,540,936,3

In [ ]:
# Análise 3: Quais regiões concentram mais crédito rural?

from pyspark.sql.functions import sum

# Volume de crédito por região
volume_regiao = (
    df_spark
    .groupBy("nomeRegiao")
    .agg(sum("VL_TOTAL").alias("VolumeTotal"))
    .orderBy("VolumeTotal", ascending=False)
)

volume_regiao.show()

# Função para formatar valores
def formatar_valor(valor):
    return f"R$ {valor/1_000_000_000:.1f} bilhões".replace(".", ",")

# Gerar texto
dados = volume_regiao.collect()

texto = (
    f"A região {dados[0]['nomeRegiao']} apresentou o maior volume de crédito rural, "
    f"totalizando {formatar_valor(dados[0]['VolumeTotal'])}. "
    f"Em seguida aparecem as regiões {dados[1]['nomeRegiao']} e "
    f"{dados[2]['nomeRegiao']}."
)

print(texto)

+------------+--------------------+
|  nomeRegiao|         VolumeTotal|
+------------+--------------------+
|         SUL|1.105407185462150...|
|CENTRO-OESTE|8.279971599362181E11|
|     SUDESTE|8.150910985503359E11|
|    NORDESTE|3.199292109565289...|
|       NORTE|2.202194824665993...|
+------------+--------------------+

A região SUL apresentou o maior volume de crédito rural, totalizando R$ 1105,4 bilhões. Em seguida aparecem as regiões CENTRO-OESTE e SUDESTE.


In [ ]:
# Análise 4: Qual modalidade recebe mais recursos?

dados = modalidades.collect()[0]

ranking = {
    "Custeio": dados["CUSTEIO"],
    "Investimento": dados["INVESTIMENTO"],
    "Comercialização": dados["COMERCIALIZACAO"],
    "Industrialização": dados["INDUSTRIALIZACAO"]
}

# Ordenar do maior para o menor
ranking_ordenado = sorted(
    ranking.items(),
    key=lambda x: x[1],
    reverse=True
)

# Função para formatar bilhões
def formatar_valor(valor):
    return f"R$ {valor/1_000_000_000:.1f} bilhões".replace(".", ",")

texto = (
    f"O ranking das modalidades de crédito rural é liderado por "
    f"{ranking_ordenado[0][0]} ({formatar_valor(ranking_ordenado[0][1])}), "
    f"seguido por {ranking_ordenado[1][0]} "
    f"({formatar_valor(ranking_ordenado[1][1])}), "
    f"{ranking_ordenado[2][0]} "
    f"({formatar_valor(ranking_ordenado[2][1])}) e "
    f"{ranking_ordenado[3][0]} "
    f"({formatar_valor(ranking_ordenado[3][1])})."
)

print(texto)

O ranking das modalidades de crédito rural é liderado por Custeio (R$ 1811,0 bilhões), seguido por Investimento (R$ 890,3 bilhões), Comercialização (R$ 415,9 bilhões) e Industrialização (R$ 171,5 bilhões).


In [ ]:
# Análise 5: Evolução Mensal

import builtins # Importa o módulo builtins para acessar a função sum nativa do Python

evolucao_mensal = (
    df_spark
    .groupBy("MesEmissao")
    .agg(sum("VL_TOTAL").alias("VolumeTotal"))
    .orderBy("MesEmissao")
)

evolucao_mensal.show()

dados = evolucao_mensal.collect()

# Ordenar do maior para o menor volume
dados_ordenados = sorted(
    dados,
    key=lambda x: x["VolumeTotal"],
    reverse=True
)

primeiro = dados_ordenados[0]
segundo = dados_ordenados[1]

# Total de crédito de todos os meses
total_geral = builtins.sum(x["VolumeTotal"] for x in dados)

# Participação percentual
perc_primeiro = (primeiro["VolumeTotal"] / total_geral) * 100
perc_segundo = (segundo["VolumeTotal"] / total_geral) * 100

texto = (
    f"Os meses com maior volume de crédito rural foram "
    f"{primeiro['MesEmissao']} ({perc_primeiro:.2f}% do total analisado) "
    f"e {segundo['MesEmissao']} ({perc_segundo:.2f}% do total analisado)."
)

print(texto)

+----------+--------------------+
|MesEmissao|         VolumeTotal|
+----------+--------------------+
|        01|1.557051900125399...|
|        02|1.746350972781403...|
|        03|2.429890865356298...|
|        04|2.376549267789904...|
|        05|2.766778725709496...|
|        06|3.437402742517486...|
|        07|2.995108569973697...|
|        08|4.114026679746501...|
|        09|3.489518305160603E11|
|        10|2.853324489125102...|
|        11|2.451089173523297...|
|        12|2.669349681909193E11|
+----------+--------------------+

Os meses com maior volume de crédito rural foram 08 (12.51% do total analisado) e 09 (10.61% do total analisado).


In [ ]:
# Análise 6: Programas de Crédito

programa = (
    df_spark
    .groupBy("cdPrograma")
    .agg(sum("VL_TOTAL").alias("VolumeTotal"))
    .orderBy("VolumeTotal", ascending=False)
)

programa.show()

+----------+--------------------+
|cdPrograma|         VolumeTotal|
+----------+--------------------+
|      0999|2.055791207894142E12|
|      0001|5.001710170241897E11|
|      0050|4.306668235985596E11|
|      0154|7.474877845932993E10|
|      0070|5.216983505508005E10|
|      0163|3.330641227266996...|
|      0156|2.470847920280999...|
|      0162|    2.20006747741E10|
|      0157|2.118552493566000...|
|      0222|1.801334207966002E10|
|      0153|1.436420396917998...|
|      0151|   1.381686774241E10|
|      0155|1.218283170778000...|
|      0152|1.032846526080000...|
|      0201| 2.010529489849999E9|
|      0160|1.8408598239599996E9|
|      0161|      9.6213487837E8|
|      0164|      2.4396089853E8|
|      0180|       1.003973076E8|
|      0165|       2.930446498E7|
+----------+--------------------+
only showing top 20 rows


In [ ]:
# Análise 7: Fontes de Recursos

fonte = (
    df_spark
    .groupBy("cdFonteRecurso")
    .agg(sum("VL_TOTAL").alias("VolumeTotal"))
    .orderBy("VolumeTotal", ascending=False)
)

fonte.show()

+--------------+--------------------+
|cdFonteRecurso|         VolumeTotal|
+--------------+--------------------+
|          0201|8.266512346108054E11|
|          0430| 6.07077983568971E11|
|          0300|5.156180050661412...|
|          0303|  3.0298676087965E11|
|          0505|2.364848484469599...|
|          0402|1.648385088071599...|
|          0502|1.411969935788698...|
|          0403| 9.75732168261699E10|
|          0503|7.797126297372006E10|
|          0501|6.992916109255013E10|
|          0431|6.103610230253001E10|
|          0800|5.214607618222005E10|
|          0506|3.894340747089999...|
|          0440|3.671485001724999E10|
|          0850|1.247050129578999...|
|          0450|1.086954592634997...|
|          0222|     7.14801563933E9|
|          0304| 6.062765294859997E9|
|          0301| 5.202112985679999E9|
|          0226|     4.01861552508E9|
+--------------+--------------------+
only showing top 20 rows
